# 07 · Checks on the final predictions

Run `python src/final.py` first. This notebook checks that the outputs look sensible before they're submitted.

## Validation predictions and the December chart values

In [1]:
import pandas as pd

preds = pd.read_csv("../validation_predictions.csv")
val = pd.read_csv("../data/validation.csv").merge(preds, on="load_id")
train = pd.read_csv("../data/train_test.csv")

# 1. Predicted rate per mile on validation vs actual rate per mile in train
train["rpm"] = train["posted_rate"] / train["distance"]
val["rpm"] = val["predicted_rate"] / val["distance"]
print(pd.concat({"train actual": train.groupby("equipment")["rpm"].median(),
                 "val predicted": val.groupby("equipment")["rpm"].median()}, axis=1).round(3))
print(val["predicted_rate"].describe().round(1))
print("corr(predicted, distance):", round(val["predicted_rate"].corr(val["distance"]), 3))

# 2. December chart values
dec = pd.read_csv("../data/december_chart_inputs.csv", parse_dates=["date"])
dec["weekday"] = dec["date"].dt.day_name()
display(dec[["date", "weekday", "predicted_rate"]])
print(dec["predicted_rate"].describe().round(1))
print("peak-to-trough, % of mean:",
      round((dec["predicted_rate"].max() - dec["predicted_rate"].min()) / dec["predicted_rate"].mean() * 100, 1))

           train actual  val predicted
equipment                             
Dry Van           2.046          2.039
Flatbed           2.220          2.206
Reefer            2.312          2.295
count    12000.0
mean      2353.5
std       1355.3
min        190.4
25%       1265.5
50%       2032.9
75%       3354.9
max       6835.3
Name: predicted_rate, dtype: float64
corr(predicted, distance): 0.992


,date,weekday,predicted_rate
0,2025-12-01,Monday,816.10
1,2025-12-02,Tuesday,823.55
2,2025-12-03,Wednesday,829.08
3,2025-12-04,Thursday,831.67
4,2025-12-05,Friday,827.62
5,2025-12-06,Saturday,819.70
6,2025-12-07,Sunday,812.08
7,2025-12-08,Monday,816.10
8,2025-12-09,Tuesday,823.55
9,2025-12-10,Wednesday,829.08


count     31.0
mean     822.8
std        6.7
min      812.1
25%      816.1
50%      823.6
75%      829.1
max      831.7
Name: predicted_rate, dtype: float64
peak-to-trough, % of mean: 2.4


## Do loads touching the 8 new cities get sensible rates?

In [2]:
unseen = ["Allentown","Charlotte","Chicago","Jackson","Knoxville","Laredo","Norfolk","San Diego"]
m = val["pickup"].isin(unseen) | val["delivery"].isin(unseen)
print(val.groupby(m)["rpm"].describe().round(3))

         count   mean    std    min    25%    50%    75%    max
False  10553.0  2.182  0.260  1.687  1.991  2.144  2.321  3.353
True    1447.0  2.190  0.262  1.718  2.005  2.143  2.336  3.296
